In [9]:
import json
import numpy as np
from tqdm import tqdm 
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

In [13]:
base_path = '/mnt/dataset/image_captioning_dataset/'
img_path = f'{base_path}FoodImages/'
splits_path = f'{base_path}DataSplit.npy'
partitions = np.load(splits_path, allow_pickle=True).item()

In [14]:
partitions['train']

array([['White Elephant Cake with Buttercream Frosting', 'white-elephant-cake-with-buttercream-frosting-240023.jpg'],
       ['Chicken Schnitzel with Capers and Parsley', 'chicken-schnitzel-with-capers-and-parsley-230977.jpg'],
       ['Our Favorite Thanksgiving Stuffing with Sausage and Cornbread', 'our-favorite-thanksgiving-stuffing-with-sausage-and-cornbread.jpg'],
       ...,
       ['Carrot Soup with Thyme and Fennel', 'carrot-soup-with-thyme-and-fennel-100977.jpg'],
       ['Quick Pork Ramen With Carrots, Zucchini, and Bok Choy', 'quick-pork-ramen-with-carrots-zucchini-and-bok-choy.jpg'],
       ['Radicchio and Plum Salad', 'radicchio-and-plum-salad.jpg']], dtype=object)

In [37]:
import easyocr
reader = easyocr.Reader(['en'], gpu=True)

def detect_text(reader,imgname):
    print(imgname)
    image = Image.open(imgname).convert('RGB')
    image = np.array(image)

    # Convert RGB to BGR (OpenCV format) for EasyOCR
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    result, free_list = reader.detect(image, slope_ths=0.1, ycenter_ths=0.5, height_ths=0.5, width_ths=0.5,text_threshold=0.9)
    horizontal_boxes = result[0]
    free_boxes = free_list[0]
    if len(horizontal_boxes) > 0 or len(free_boxes) > 0:
        image_with_boxes = image.copy()
        
        # Draw bounding boxes for horizontal text (green rectangles)
        for box in horizontal_boxes:
            x1, x2, y1, y2 = map(int, box)
            cv2.rectangle(image_with_boxes, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Draw bounding boxes for rotated text (red quadrilaterals)
        for box in free_boxes:
            points = np.array(box, dtype=np.int32).reshape(-1, 2)
            cv2.polylines(image_with_boxes, [points], isClosed=True, color=(0, 0, 255), thickness=2)
        
        # Convert BGR back to RGB for Matplotlib
        image_with_boxes_rgb = cv2.cvtColor(image_with_boxes, cv2.COLOR_BGR2RGB)
        
        # Display the image with bounding boxes
        plt.figure(figsize=(8, 6))
        plt.imshow(image_with_boxes_rgb)
        plt.title("Image with Detected Text Regions\n(Green: Horizontal, Red: Rotated)")
        plt.axis('off')
        plt.show()
        return True
    else:
        return False

In [ ]:
filtered_partition = {}
noisy_data = []
for k in partitions.keys():
    filtered_data = []
    for title, imgname in partitions[k]:
        has_text = detect_text(reader,img_path+imgname)
        if not has_text:
            filtered_data.append([title, img_path+imgname])
        else:
            noisy_data.append([title, img_path+imgname])
    filtered_partition[k] = np.array(filtered_data)

In [42]:
np.array(filtered_data).shape

(13197, 2)

In [44]:
np.array(noisy_data).shape

(266, 2)

In [47]:
np.save(f'{base_path}FilteredDataSplit.npy', filtered_partition)